# Stratified K-Fold and Grid Search from Scratch

Plain k-fold (see `Cross Validation.ipynb`) shuffles samples blindly. On imbalanced data a
fold can end up with very few minority samples, or none, so its score is meaningless.
Stratified k-fold deals each class across the folds separately so every fold keeps the
overall class ratio.

On top of that this notebook builds `grid_search_cv`, which tries every combination in a
parameter grid, scores each with cross-validation and returns the best one. The test set is
only touched once at the end.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

## `stratified_k_fold_split`

For each class, shuffle its indices and spread them round-robin over the `k` folds. Per-class
fold sizes differ by at most one, and each fold's class ratio matches the full dataset.

In [ ]:
def stratified_k_fold_split(y, k=5, shuffle=True, seed=None):
    y = np.asarray(y)
    rng = np.random.RandomState(seed)
    fold_members = [[] for _ in range(k)]
    offset = 0
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        if shuffle:
            rng.shuffle(idx)
        for j, i in enumerate(idx):
            # offset rotates the starting fold per class so small classes do not pile into fold 0
            fold_members[(j + offset) % k].append(i)
        offset += len(idx)
    all_idx = np.arange(len(y))
    folds = []
    for members in fold_members:
        val_idx = np.sort(np.array(members, dtype=int))
        train_idx = np.setdiff1d(all_idx, val_idx)
        folds.append((train_idx, val_idx))
    return folds


# Imbalanced toy labels: 90 negatives, 10 positives
y_toy = np.array([0] * 90 + [1] * 10)
for tr, val in stratified_k_fold_split(y_toy, k=5, seed=0):
    assert y_toy[val].sum() == 2 and len(val) == 20
    assert len(set(tr) & set(val)) == 0
print('stratified_k_fold_split verified: each fold has 18 negatives + 2 positives.')

## Why it matters

Take breast cancer and keep only 15 of the malignant samples to make the data heavily
imbalanced. Compare how much the minority share varies per validation fold.

In [ ]:
data = load_breast_cancer()
X_all, y_all = data.data, data.target
rng = np.random.RandomState(0)
neg = np.where(y_all == 0)[0]
pos = np.where(y_all == 1)[0]
keep = np.concatenate([rng.choice(neg, 15, replace=False), pos])
X_imb, y_imb = X_all[keep], y_all[keep]
print('minority share overall: %.3f' % np.mean(y_imb == 0))


def plain_k_fold(n, k, seed):
    idx = np.random.RandomState(seed).permutation(n)
    parts = np.array_split(idx, k)
    return [(np.setdiff1d(idx, p), p) for p in parts]


plain_share, strat_share = [], []
for seed in range(20):
    plain_share += [np.mean(y_imb[v] == 0) for _, v in plain_k_fold(len(y_imb), 5, seed)]
    strat_share += [np.mean(y_imb[v] == 0) for _, v in stratified_k_fold_split(y_imb, 5, seed=seed)]

print('plain      std of minority share per fold: %.4f' % np.std(plain_share))
print('stratified std of minority share per fold: %.4f' % np.std(strat_share))
plt.hist(plain_share, bins=15, alpha=0.6, label='plain')
plt.hist(strat_share, bins=15, alpha=0.6, label='stratified')
plt.xlabel('minority share in validation fold'); plt.legend(); plt.show()

## `grid_search_cv`

`make_fit_predict(**params)` returns a `fit_predict(X_train, y_train, X_val)` function, so any
model from this repo can be tuned. The search expands the grid with `itertools.product`,
evaluates each combination on the *same* stratified folds (so differences come from the
parameters, not fold luck) and records mean and std of the score.

In [ ]:
def grid_search_cv(X, y, param_grid, make_fit_predict, k=5, seed=0,
                   metric_fn=lambda yt, yp: np.mean(yt == yp)):
    names = list(param_grid)
    folds = stratified_k_fold_split(y, k=k, seed=seed)
    results = []
    for values in itertools.product(*(param_grid[n] for n in names)):
        params = dict(zip(names, values))
        fit_predict = make_fit_predict(**params)
        scores = np.array([metric_fn(y[v], fit_predict(X[t], y[t], X[v])) for t, v in folds])
        results.append({'params': params, 'mean': scores.mean(), 'std': scores.std()})
    best = max(results, key=lambda r: r['mean'])
    return best, results


print('grid_search_cv defined.')

## Model to tune

A from-scratch KNN (hyperparameters: `n_neighbors`, distance weighting) on standardised
features. Scaling statistics come from the training fold only, to avoid leaking validation data.

In [ ]:
def make_knn(n_neighbors, weighted):
    def fit_predict(X_tr, y_tr, X_val):
        mu, sd = X_tr.mean(axis=0), X_tr.std(axis=0) + 1e-12
        A, B = (X_tr - mu) / sd, (X_val - mu) / sd
        d = np.sqrt(((B[:, None, :] - A[None, :, :]) ** 2).sum(axis=2))
        nn = np.argsort(d, axis=1)[:, :n_neighbors]
        preds = []
        for row, idx in enumerate(nn):
            w = 1.0 / (d[row, idx] + 1e-9) if weighted else np.ones(len(idx))
            votes = {}
            for lbl, wt in zip(y_tr[idx], w):
                votes[lbl] = votes.get(lbl, 0.0) + wt
            preds.append(max(votes, key=votes.get))
        return np.array(preds)
    return fit_predict


X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=1)
grid = {'n_neighbors': [1, 3, 5, 9, 15, 25], 'weighted': [False, True]}
best, results = grid_search_cv(X_tr, y_tr, grid, make_knn, k=5)
for r in sorted(results, key=lambda r: -r['mean'])[:5]:
    print(r['params'], '%.4f +/- %.4f' % (r['mean'], r['std']))
print('best:', best['params'])

## Final check on the held-out test set

The test split was never used during the search.

In [ ]:
final = make_knn(**best['params'])
test_acc = np.mean(final(X_tr, y_tr, X_te) == y_te)
print('CV accuracy  : %.4f' % best['mean'])
print('Test accuracy: %.4f' % test_acc)

ks = grid['n_neighbors']
for w in (False, True):
    ms = [r['mean'] for r in results if r['params']['weighted'] == w]
    plt.plot(ks, ms, marker='o', label='weighted' if w else 'uniform')
plt.xlabel('n_neighbors'); plt.ylabel('CV accuracy'); plt.legend(); plt.show()

## Check against scikit-learn

In [ ]:
from sklearn.model_selection import StratifiedKFold
overall = np.mean(y_imb == 0)
for name, folds in [('sklearn', list(StratifiedKFold(5, shuffle=True, random_state=0).split(X_imb, y_imb))),
                    ('scratch', stratified_k_fold_split(y_imb, 5, seed=0))]:
    shares = [np.mean(y_imb[v] == 0) for _, v in folds]
    assert max(abs(s - overall) for s in shares) < 0.05, name
    print(name, np.round(shares, 3))
print('Both keep the minority share close to the overall ratio in every fold.')